# Environment Setup

In [8]:
from inspect import getsourcelines, getsource
import hashlib

def md5_source(obj):
    src = ''.join(getsourcelines(obj)[0])
    return hashlib.md5(src.encode()).hexdigest()

print("BTUNet source hash:", md5_source(BTUNet))


OSError: source code not available

In [133]:
!pip install albumentations
!pip install timm
!pip install pandas
!pip install matplotlib


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [134]:
# Cell 1: Environment Setup
# %matplotlib inline # Uncomment if running in a different environment

!pip -q install einops timm torchmetrics  # already in most fastMRI envs

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import os # For os.makedirs

# Set random seeds for reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip
Using device: cuda


# Dataset Class Definition

In [135]:
# Cell 2: Dataset Class Definition (Modified for Augmentation)
import torch
from torch.utils.data import Dataset
import os
from pathlib import Path
import numpy as np # Needed for Albumentations
import albumentations as A
# from albumentations.pytorch import ToTensorV2 # Not strictly needed if data is already tensors and handled manually

class ProcessedFastMRIDataset(Dataset):
    """Dataset for loading preprocessed FastMRI data or creating from raw files"""
    
    def __init__(self, data_dir=None, file_list=None, mode='train', mask_func=None, use_processed=True, no_aug_chance=0.0): # Added no_aug_chance
        self.mode = mode
        self.use_processed = use_processed
        self.no_aug_chance = no_aug_chance # Chance to skip augmentation for a sample
        
        if use_processed:
            self.data_dir = os.path.join(data_dir, mode) # [cite: 1]
            try:
                all_files_in_dir = os.listdir(self.data_dir) # [cite: 1]
            except FileNotFoundError:
                raise FileNotFoundError(f"Data directory not found: {self.data_dir}")

            all_pt_files = sorted([os.path.join(self.data_dir, f) for f in all_files_in_dir if f.endswith('.pt')]) # [cite: 1]

            self.batch_files = [] # [cite: 1]
            for f_path_str in all_pt_files:
                if "metadata" not in Path(f_path_str).name: # [cite: 1]
                    self.batch_files.append(f_path_str) # [cite: 1]
            
            if not self.batch_files: # [cite: 1]
                raise FileNotFoundError(f"No valid .pt files (after filtering 'metadata' files) found in {self.data_dir}")
                
            self.examples = [] # [cite: 1]
            
            for i, batch_file_path in enumerate(self.batch_files): # [cite: 1]
                try:
                    batch_peek = torch.load(batch_file_path, map_location='cpu') # [cite: 1]

                    if not isinstance(batch_peek, dict): # [cite: 1]
                        print(f"  Warning: Skipped {batch_file_path}. Loaded object is not a dictionary (type: {type(batch_peek)}).")
                        continue
                    if 'inputs' not in batch_peek: # [cite: 1]
                        print(f"  Warning: Skipped {batch_file_path}. Missing 'inputs' key. Keys present: {list(batch_peek.keys())}.")
                        continue
                    
                    num_samples = len(batch_peek['inputs']) # [cite: 1]
                    self.examples.extend([(i, j) for j in range(num_samples)]) # [cite: 1]
                    del batch_peek # [cite: 1]

                except Exception as e:
                    print(f"  Error peaking into file {batch_file_path}: {e}. Skipping.")
            
            if not self.examples: # [cite: 1]
                raise ValueError(f"No valid examples could be loaded from {self.data_dir}. Check warnings above.")
            
            print(f"Successfully indexed a total of {len(self.examples)} examples from {len(self.batch_files)} .pt files in {self.data_dir}.")

        else:
            raise NotImplementedError("Raw file processing not fully set up in this tuning script example.") # [cite: 1]
        
        # Define augmentation pipelines
        self.TARGET_HEIGHT = 320  # Example, adjust if needed
        self.TARGET_WIDTH = 320   # Example, adjust if needed

        if self.mode == 'train':
            self.geometric_transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.Rotate(limit=10, p=0.3, border_mode=0, interpolation=1), # Use cv2.INTER_LINEAR for interpolation
                A.RandomScale(scale_limit=0.1, p=0.2, interpolation=1),
                # Add PadIfNeeded and RandomCrop (or CenterCrop) to ensure fixed size
                A.PadIfNeeded(min_height=self.TARGET_HEIGHT, min_width=self.TARGET_WIDTH, border_mode=0, p=1.0),
                A.RandomCrop(height=self.TARGET_HEIGHT, width=self.TARGET_WIDTH, p=1.0),
            ])
            self.intensity_transform = A.Compose([
                 A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
                 # Ensure A.GaussNoise parameters are correct for your albumentations version
                 # If 'var_limit' gives warnings, check help(A.GaussNoise) for your version's API
                 A.GaussNoise(
                    std_range=(0.01, 0.05),         # Choose a reasonable std dev range as a fraction of max value
                    mean_range=(0.0, 0.0),          # Zero mean by default
                    per_channel=True,
                    noise_scale_factor=1.0,
                    p=0.2
                )
            ])
        else: # For 'val' or 'test' mode
            # Ensure validation/test data is also consistently sized, typically via CenterCrop
            self.geometric_transform = A.Compose([
                A.PadIfNeeded(min_height=self.TARGET_HEIGHT, min_width=self.TARGET_WIDTH, border_mode=0, p=1.0),
                A.CenterCrop(height=self.TARGET_HEIGHT, width=self.TARGET_WIDTH, p=1.0),
            ]) # Or set to None if your preprocessed val data is already correctly sized
            self.intensity_transform = None

    def __len__(self):
        return len(self.examples) # [cite: 1]
    
    def __getitem__(self, idx):
        if self.use_processed: # [cite: 1]
            batch_idx, sample_idx = self.examples[idx] # [cite: 1]
            batch_data = torch.load(self.batch_files[batch_idx], map_location='cpu') # [cite: 1]
            inputs_tensor = batch_data['inputs'][sample_idx] # [cite: 1]
            targets_tensor = batch_data['targets'][sample_idx] # [cite: 1]
            del batch_data # [cite: 1]
        else:
            raise NotImplementedError("Raw file processing not fully set up in this tuning script example.") # [cite: 1]
        
        # Ensure tensors are 2D [H, W] before converting to NumPy
        if inputs_tensor.ndim > 2: inputs_tensor = inputs_tensor.squeeze()
        if targets_tensor.ndim > 2: targets_tensor = targets_tensor.squeeze()

        inputs_np = inputs_tensor.numpy()
        targets_np = targets_tensor.numpy()

        # Apply augmentations only in 'train' mode and not skipped by chance
        if self.mode == 'train' and random.random() > self.no_aug_chance:
            if self.geometric_transform:
                # Albumentations expects 'image' and 'mask' keys
                augmented = self.geometric_transform(image=inputs_np, mask=targets_np)
                inputs_np = augmented['image']
                targets_np = augmented['mask']
            
            if self.intensity_transform:
                # Intensity transforms only on the input
                augmented_input = self.intensity_transform(image=inputs_np)
                inputs_np = augmented_input['image']

        # Convert back to Tensors
        # The model expects [C, H, W], so add channel dimension if not present
        inputs = torch.from_numpy(inputs_np).unsqueeze(0) # Add channel dim: [1, H, W]
        targets = torch.from_numpy(targets_np).unsqueeze(0) # Add channel dim: [1, H, W]
        
        return inputs, targets

# Loss Functions and Metrics

In [136]:
# Cell 3: Loss Functions and Metrics (from Swin UNET Overfit.ipynb)
import cv2 # for albumentations border_mode and interpolation constants

class SSIMLoss(nn.Module):
    """SSIM loss module for MRI reconstruction"""
    def __init__(self, win_size=7, k1=0.01, k2=0.03, data_range=1.0): # Added data_range
        super().__init__()
        self.win_size = win_size
        self.k1, self.k2 = k1, k2
        self.data_range = data_range # Max value of the input images
        # Create a non-trainable buffer for the window
        self.register_buffer('w', torch.ones(1, 1, win_size, win_size) / win_size**2)
        self.cov_norm = win_size**2 / (win_size**2 - 1) # Correction factor for variance

    def forward(self, x, y): # Expects x, y to be 4D tensors [B, C, H, W]
        # Ensure inputs are 4D
        if x.ndim == 3: x = x.unsqueeze(1)
        if y.ndim == 3: y = y.unsqueeze(1)
        
        # Automatic data_range detection if not specified (or set to None)
        # For this script, we assume normalized data [0,1], so data_range=1.0 is fine.
        # If your data isn't normalized, you might want to pass a different data_range
        # or dynamically calculate it (though this adds overhead).
        # For simplicity and consistency with previous notebooks, we'll use a fixed data_range.
        
        C1 = (self.k1 * self.data_range)**2
        C2 = (self.k2 * self.data_range)**2
        
        ux = F.conv2d(x, self.w)
        uy = F.conv2d(y, self.w)
        
        uxx = F.conv2d(x * x, self.w)
        uyy = F.conv2d(y * y, self.w)
        uxy = F.conv2d(x * y, self.w)
        
        vx = self.cov_norm * (uxx - ux * ux)
        vy = self.cov_norm * (uyy - uy * uy)
        vxy = self.cov_norm * (uxy - ux * uy)
        
        A1 = 2 * ux * uy + C1
        A2 = 2 * vxy + C2
        B1 = ux**2 + uy**2 + C1
        B2 = vx + vy + C2
        
        D = B1 * B2
        S = (A1 * A2) / D
        
        # When D is close to zero, SSIM can be unstable. Clamp S to avoid NaN/Inf.
        # This usually happens when image patches are flat.
        S = torch.clamp(S, min=0, max=1) # Or handle based on skimage's behavior for such cases
        
        return 1 - S.mean()

class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.84, ssim_data_range=1.0): # Pass ssim_data_range
        super().__init__()
        self.alpha = alpha
        self.l1_loss = nn.L1Loss()
        self.ssim_loss = SSIMLoss(data_range=ssim_data_range)
        
    def forward(self, pred, target):
        l1 = self.l1_loss(pred, target)
        ssim = self.ssim_loss(pred, target)
        return self.alpha * l1 + (1 - self.alpha) * ssim

def calculate_psnr_torch(img1, img2, data_range=1.0): # Added data_range
    """Calculate PSNR between two PyTorch tensors"""
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0: return torch.tensor(float('inf')).to(img1.device) # Perfect match
    return 20 * torch.log10(data_range / torch.sqrt(mse))

def calculate_ssim_skimage(img1_np, img2_np, data_range=None): # Explicitly for numpy arrays
    """Calculate SSIM between two numpy arrays using scikit-image"""
    # Ensure images are single channel if they are multi-channel grayscale
    if img1_np.ndim == 3 and img1_np.shape[0] == 1: img1_np = img1_np.squeeze(0)
    if img2_np.ndim == 3 and img2_np.shape[0] == 1: img2_np = img2_np.squeeze(0)
    
    # If data_range is not provided, scikit-image will estimate it.
    # It's better to provide it if known (e.g., 1.0 for normalized [0,1] images)
    current_data_range = data_range if data_range is not None else img1_np.max() - img1_np.min()
    if current_data_range == 0: # Avoid division by zero if image is flat
        return 1.0 if np.allclose(img1_np, img2_np) else 0.0

    return ssim(img1_np, img2_np, data_range=current_data_range, channel_axis=None, win_size=7)

# Common Model Blocks

In [4]:
# Cell 4: Common Model Blocks (from Transformer Variant Tests (4).ipynb)
class DoubleConv(nn.Sequential):
    def __init__(self, in_ch, out_ch):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True), nn.GELU()
        )

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool=nn.AvgPool2d(2)
        self.conv=DoubleConv(in_ch,out_ch)
    def forward(self,x): return self.conv(self.pool(x))

class Up(nn.Module):
    def __init__(self,in_ch,out_ch): # Input channels here refers to (skip_ch + upsampled_ch)
        super().__init__()
        # The input to DoubleConv will be cat([upsampled_x, skip]), so in_ch for DoubleConv
        # should be out_ch_from_upsample (which is in_ch for Up) + skip_ch.
        # However, the UNet structure passes pre-calculated channel numbers to Up.
        # Example: Up(ch*2, ch) means up(in_ch=ch*2, out_ch=ch), and it receives x from bottleneck (ch*2)
        # and skip from encoder (ch). So Conv2d input will be ch*2.
        # The `in_ch` to `Up` module is actually the sum of channels after concatenation.
        self.up=nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False)
        self.conv=DoubleConv(in_ch,out_ch) # DoubleConv input is in_ch, output is out_ch

    def forward(self, x, skip):
        x = self.up(x) # Upsamples x
    
        # Spatial size guard
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
    
        # Concatenate along channel dimension
        # x comes from a deeper layer (e.g. 512 channels), skip from encoder (e.g. 256 channels)
        # After upsampling, x might still be 512 channels.
        # The DoubleConv in Up(in_ch, out_ch) expects in_ch = x.shape[1] + skip.shape[1]
        # The UNet definition usually handles this by specifying in_ch to Up correctly.
        # E.g., self.ups.append(Up(ch + ch//2, ch//2)) if ch=512, skip is 256 (ch//2)
        # Then in_ch to Up's DoubleConv will be (512 from upsampled x) + (256 from skip)
        # This means self.conv = DoubleConv(in_ch_x_upsampled + in_ch_skip, out_ch_final)
        
        # The original `Up` in Transformer Variant Tests used `DoubleConv(in_ch, out_ch)`.
        # In the UNet forward: `Up(ch*2, ch)` -> `DoubleConv(ch*2, ch)` meaning input after cat is ch*2.
        # This implies that input x to Up had ch channels, and skip had ch channels.
        # up(ch, ch) then cat(ch, ch) -> DoubleConv(2*ch, ch)
        # If `Up(512, 256)` -> `DoubleConv(512, 256)` expects input of 512.
        # If x is 256 (after upsample from 128), skip is 256. cat(256, 256) = 512. This matches.

        return self.conv(torch.cat([x, skip], dim=1))


# Transformer Primitives

In [138]:

# Transformer Primitives
class CustomMLP(nn.Sequential):
    def __init__(self,dim,mlp_dim=None,p=0.):
        mlp_dim = mlp_dim or dim*4
        super().__init__(nn.Linear(dim,mlp_dim),nn.GELU(),nn.Dropout(p),
                         nn.Linear(mlp_dim,dim),nn.Dropout(p))

class TransformerBlock(nn.Module):
    def __init__(self,dim,heads=8,p=0.): # p is dropout probability
        super().__init__()
        self.norm1=nn.LayerNorm(dim)
        self.attn=nn.MultiheadAttention(dim,heads,dropout=p,batch_first=True)
        self.norm2=nn.LayerNorm(dim)
        self.mlp=CustomMLP(dim,p=p) # Pass dropout to CustomMLP as well
    def forward(self,x): # x is (B, N, C) where N is sequence length, C is dim
        x_res = x
        x = self.norm1(x)
        attn_out, _ = self.attn(x,x,x) # Query, Key, Value are all x
        x = x_res + attn_out # First residual connection
        
        x_res = x
        x = self.norm2(x)
        mlp_out = self.mlp(x)
        x = x_res + mlp_out # Second residual connection
        return x

# Swin-UNet

In [139]:
# Cell 5: Swin-UNet Architecture
from timm.models.swin_transformer import WindowAttention, Mlp as TimmMLP # Use a distinct name
from timm.models.layers import DropPath

def window_partition(x, window_size: int):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows

def window_reverse(windows, window_size: int, H: int, W: int):
    num_windows_h = H // window_size
    num_windows_w = W // window_size
    B = windows.shape[0] // (num_windows_h * num_windows_w)
    x = windows.view(B, num_windows_h, num_windows_w, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x
    
class SwinBlock(nn.Module):
    def __init__(self, dim, ws=8, heads=4, drop_path_rate=0.): # Added drop_path_rate
        super().__init__()
        self.dim = dim
        self.window_size = ws
        self.heads = heads

        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(
            dim,
            num_heads=heads,
            window_size=(ws, ws),
            qkv_bias=True
        )
        self.drop_path1 = DropPath(drop_path_rate) if drop_path_rate > 0. else nn.Identity()
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = TimmMLP(in_features=dim, hidden_features=int(dim * 4), act_layer=nn.GELU, drop=0.1) # TimmMLP uses 'drop' for dropout
        self.drop_path2 = DropPath(drop_path_rate) if drop_path_rate > 0. else nn.Identity()


    def forward(self, x): # Expected input x: (B, C, H, W)
        B, C, H, W = x.shape
        if C != self.dim:
            raise ValueError(f"Input channel dimension {C} does not match block dimension {self.dim}")

        x_spatial = rearrange(x, 'b c h w -> b h w c')
        shortcut = x_spatial 

        x_normed = self.norm1(x_spatial)

        H_pad, W_pad = H, W
        pad_l = pad_t = 0
        pad_r = (self.window_size - W % self.window_size) % self.window_size
        pad_b = (self.window_size - H % self.window_size) % self.window_size
        if pad_r > 0 or pad_b > 0:
            x_normed = F.pad(x_normed, (0, 0, pad_l, pad_r, pad_t, pad_b)) 
            H_pad += pad_b
            W_pad += pad_r
        
        x_windows = window_partition(x_normed, self.window_size)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)
        
        # W-MSA
        # For SW-MSA, cyclic shift and attention mask would be needed.
        attn_windows = self.attn(x_windows, mask=None)
        
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        merged_windows = window_reverse(attn_windows, self.window_size, H_pad, W_pad)

        if pad_r > 0 or pad_b > 0:
            merged_windows = merged_windows[:, :H, :W, :].contiguous()
        
        x_spatial = shortcut + self.drop_path1(merged_windows) # If using DropPath
        # x_spatial = shortcut + merged_windows
        
        x_ffn_input = self.norm2(x_spatial)
        x_ffn = self.mlp(x_ffn_input)
        
        x_out_spatial = x_spatial + self.drop_path2(x_ffn) # If using DropPath
        # x_out_spatial = x_spatial + x_ffn

        x_out = rearrange(x_out_spatial, 'b h w c -> b c h w')
        
        return x_out

class SwinUNet(nn.Module):
    def __init__(self, in_chans=1, out_chans=1, base=32, ws=8, heads_per_stage=(4,4,4,4),
                 drop_path_rate_max=0.1): # Add max drop_path_rate
        super().__init__()
        chs=[base, base*2, base*4, base*8]
        self.inc=nn.Conv2d(in_chans,chs[0],3,padding=1)

        num_total_swin_blocks = 7 # s1,s2,s3,s4, su3,su2,su1
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate_max, num_total_swin_blocks)]  # Stoch. depth rule

        # Encoder stages
        self.s1=self._stage(chs[0],chs[0],ws, heads=heads_per_stage[0], current_dpr=dpr[0])
        self.d1=nn.Conv2d(chs[0],chs[1],kernel_size=2,stride=2)
        self.s2=self._stage(chs[1],chs[1],ws, heads=heads_per_stage[1], current_dpr=dpr[1])
        self.d2=nn.Conv2d(chs[1],chs[2],kernel_size=2,stride=2)
        self.s3=self._stage(chs[2],chs[2],ws, heads=heads_per_stage[2], current_dpr=dpr[2])
        self.d3=nn.Conv2d(chs[2],chs[3],kernel_size=2,stride=2)
        self.s4=self._stage(chs[3],chs[3],ws, heads=heads_per_stage[3], current_dpr=dpr[3]) # Bottleneck

        # Decoder stages
        self.u3=nn.ConvTranspose2d(chs[3],chs[2],kernel_size=2,stride=2)
        self.su3=self._stage(chs[2]*2,chs[2],ws, heads=heads_per_stage[2], current_dpr=dpr[4])
        self.u2=nn.ConvTranspose2d(chs[2],chs[1],kernel_size=2,stride=2)
        self.su2=self._stage(chs[1]*2,chs[1],ws, heads=heads_per_stage[1], current_dpr=dpr[5])
        self.u1=nn.ConvTranspose2d(chs[1],chs[0],kernel_size=2,stride=2)
        self.su1=self._stage(chs[0]*2,chs[0],ws, heads=heads_per_stage[0], current_dpr=dpr[6])

        self.out=nn.Conv2d(chs[0],out_chans,kernel_size=1)

    def _stage(self,in_ch,out_ch,ws,heads, current_dpr=0.): # Accept current_dpr
        return nn.Sequential(
            nn.Conv2d(in_ch,out_ch,kernel_size=3,padding=1,bias=False),
            nn.InstanceNorm2d(out_ch,affine=True),
            nn.GELU(),
            SwinBlock(dim=out_ch, ws=ws, heads=heads, drop_path_rate=current_dpr) # Pass dpr
        )
    def forward(self,x):
        x0 = self.inc(x) # Initial conv
        s1 = self.s1(x0) # Stage 1 + skip
        
        s2_in = self.d1(s1) # Downsample
        s2 = self.s2(s2_in) # Stage 2 + skip
        
        s3_in = self.d2(s2) # Downsample
        s3 = self.s3(s3_in) # Stage 3 + skip
        
        b_in = self.d3(s3)  # Downsample to bottleneck
        b = self.s4(b_in)   # Bottleneck stage
        
        u3_up = self.u3(b)  # Upsample
        u3 = self.su3(torch.cat([u3_up, s3], dim=1)) # Concatenate skip and process
        
        u2_up = self.u2(u3) # Upsample
        u2 = self.su2(torch.cat([u2_up, s2], dim=1)) # Concatenate skip and process
        
        u1_up = self.u1(u2) # Upsample
        u1 = self.su1(torch.cat([u1_up, s1], dim=1)) # Concatenate skip and process
        
        return self.out(u1)

# Generic Trainer

In [5]:
# Cell 5: UNet Baseline (Core for BT-UNet)
class UNet(nn.Module):
    def __init__(self, in_chans=1, out_chans=1, chans=32, num_pool_layers=4):
        super().__init__()
        self.in_chans = in_chans
        self.out_chans = out_chans
        self.num_pool_layers = num_pool_layers

        self.inc = DoubleConv(in_chans, chans)
        
        # Store the output channels of each DoubleConv block in the encoder path
        # These will be used for skip connections
        encoder_feature_channels = [chans] 
        
        self.downs = nn.ModuleList()
        current_encoder_ch = chans # Input channels to the first Down module's DoubleConv
        for _ in range(num_pool_layers):
            self.downs.append(Down(current_encoder_ch, current_encoder_ch * 2))
            current_encoder_ch *= 2 # Output channels of this Down module's DoubleConv
            encoder_feature_channels.append(current_encoder_ch) 
        # After loop, current_encoder_ch = chans * 2^num_pool_layers (e.g., 32 * 16 = 512 for chans=32, num_pool_layers=4)
        # encoder_feature_channels = [32, 64, 128, 256, 512]

        # Bottleneck takes the output of the last Down layer
        self.bottleneck_conv = DoubleConv(current_encoder_ch, current_encoder_ch * 2) 
        
        # Decoder
        # current_decoder_ch starts as the output channels of the bottleneck
        current_decoder_ch = current_encoder_ch * 2 # (e.g., 1024)

        self.ups = nn.ModuleList()
        # Skips are taken from encoder_feature_channels in reverse order,
        # excluding the last one (which was input to bottleneck).
        # So, skips are from encoder_feature_channels[num_pool_layers-1], ..., encoder_feature_channels[0]
        # Or, using negative indexing: encoder_feature_channels[-2], [-3], ..., up to the one from self.inc
        
        for i in range(num_pool_layers):
            # Determine the number of channels from the corresponding skip connection in the encoder
            # The skip connection for the i-th Up layer (from deepest, i=0) corresponds to
            # the (num_pool_layers - 1 - i)-th element in encoder_feature_channels list,
            # or more simply, encoder_feature_channels[-(i + 2)] using negative indexing.
            skip_ch_for_this_level = encoder_feature_channels[-(i + 2)] 
            
            # The Up module's DoubleConv takes (channels_from_upsampled_deeper_layer + channels_from_skip)
            # The output channels of this Up stage's DoubleConv is typically skip_ch_for_this_level
            output_ch_of_this_up_stage = skip_ch_for_this_level
            
            # Define the Up module: Up(in_channels_for_DoubleConv, out_channels_for_DoubleConv)
            # in_channels_for_DoubleConv = current_decoder_ch (from upsampled deeper layer) + skip_ch_for_this_level
            self.ups.append(Up(current_decoder_ch + skip_ch_for_this_level, output_ch_of_this_up_stage))
            
            # Update current_decoder_ch for the next (shallower) Up layer
            current_decoder_ch = output_ch_of_this_up_stage 
        
        # The final convolution maps the channels from the last Up layer to out_chans
        # After the loop, current_decoder_ch should be equal to 'chans' (the output of self.inc)
        self.outc = nn.Conv2d(current_decoder_ch, out_chans, kernel_size=1) 

    def forward(self, x):
        # Encoder: Store outputs of each DoubleConv block for skip connections
        encoder_outputs = [] 
        enc_x = self.inc(x)
        encoder_outputs.append(enc_x) 
        for down_module in self.downs:
            enc_x = down_module(enc_x) 
            encoder_outputs.append(enc_x)
        # encoder_outputs list now contains:
        # [output_of_inc, output_of_downs[0].conv, ..., output_of_downs[num_pool_layers-1].conv]

        # Bottleneck: Takes the output of the last Down layer
        b = self.bottleneck_conv(encoder_outputs[-1])

        # Decoder
        dec_x = b
        for i, up_module in enumerate(self.ups):
            # Retrieve the corresponding skip connection from the encoder_outputs list
            # For the i-th Up module (from deepest, i=0), the skip connection is
            # encoder_outputs[num_pool_layers - 1 - i] or encoder_outputs[-(i + 2)]
            skip_connection = encoder_outputs[-(i + 2)] 
            dec_x = up_module(dec_x, skip_connection)
            
        # Final output layer
        out = self.outc(dec_x)
        
        # Ensure output spatial dimensions match input if necessary (though Up and Down should handle this)
        if out.shape[-2:] != x.shape[-2:]:
            out = F.interpolate(out, size=x.shape[-2:], mode='bilinear', align_corners=False)
            
        return out


# BT UNet

In [7]:
# Cell 6: BT-UNet Model Definition (from Transformer Variant Tests (4).ipynb)
class BTUNet(nn.Module):
    def __init__(self, in_chans=1, out_chans=1, base_channels=32, num_pool_layers=4, 
                 tr_depth=4, tr_heads=8, tr_dropout=0.1): # Added tr_dropout
        super().__init__()
        
        # Define the U-Net core
        # The UNet's 'chans' parameter is BTUNet's 'base_channels'
        self.core_unet = UNet(in_chans=in_chans, out_chans=out_chans, # out_chans for UNet core is not final out_chans
                               chans=base_channels, num_pool_layers=num_pool_layers)

        # Encoder path (re-use from core_unet for clarity and direct access if needed)
        self.inc = self.core_unet.inc
        self.downs = self.core_unet.downs
        
        # Determine channel dimension for TransformerBlock based on UNet's structure
        # After 'num_pool_layers' downsamplings, channels become base_channels * (2^num_pool_layers)
        # The UNet's bottleneck_conv takes this and outputs base_channels * (2^(num_pool_layers+1))
        # This output of bottleneck_conv is what we replace with Transformer blocks.
        # So, C for TransformerBlock is base_channels * (2^num_pool_layers)
        
        # Example: base_channels=32, num_pool_layers=4
        # inc: 32 -> 32
        # d0: 32 -> 64 (H/2)
        # d1: 64 -> 128 (H/4)
        # d2: 128 -> 256 (H/8)
        # d3: 256 -> 512 (H/16) <- This is the input to the Transformer bottleneck
        # So, C = base_channels * (2**num_pool_layers)
        
        C_transformer = base_channels * (2**num_pool_layers)
        
        self.tr_blocks = nn.Sequential(
            *[TransformerBlock(dim=C_transformer, heads=tr_heads, p=tr_dropout) for _ in range(tr_depth)]
        )

        # Decoder path (re-use from core_unet)
        # The UNet's Up layers expect specific channel inputs after concatenation.
        # The output of tr_blocks (C_transformer channels) will be the 'x' input to the first Up layer.
        # The skip connection to this first Up layer comes from the last Down layer BEFORE the bottleneck.
        # UNet.ups[0] = Up(in_ch_from_prev_up + in_ch_from_skip, out_ch_current_up)
        # For the first Up layer after Transformer:
        #   in_ch_from_prev_up = C_transformer (output of tr_blocks)
        #   in_ch_from_skip = base_channels * (2**(num_pool_layers-1)) (output of (num_pool_layers-1)-th Down block)
        # This logic is handled within the UNet.ups definition if core_unet.bottleneck_conv is effectively bypased
        # and its output dimensions match C_transformer.

        # Let's adjust how UNet is used:
        # 1. Encoder part up to the layer before original bottleneck_conv
        # 2. Transformer blocks
        # 3. Decoder part, taking output of Transformer and skips from encoder

        # Re-defining UNet parts for BT-UNet structure:
        self.encoder_inc = self.core_unet.inc
        self.encoder_downs = nn.ModuleList() # To store (num_pool_layers) Down blocks
        
        current_ch = base_channels
        for _ in range(num_pool_layers):
            self.encoder_downs.append(Down(current_ch, current_ch * 2))
            current_ch *= 2
        # After this loop, current_ch = base_channels * (2**num_pool_layers) which is C_transformer

        self.transformer_bottleneck = nn.Sequential(
            *[TransformerBlock(dim=C_transformer, heads=tr_heads, p=tr_dropout) for _ in range(tr_depth)]
        )
        
        # Decoder
        self.decoder_ups = nn.ModuleList()
        # Input to first Up layer is C_transformer from Transformer + C_transformer//2 from last encoder skip
        # Output from first Up layer is C_transformer//2
        # So, Up(C_transformer + C_transformer//2, C_transformer//2)

        # current_ch is C_transformer (e.g. 512 if base=32, layers=4)
        for _ in range(num_pool_layers):
            # Skip connection comes from encoder_downs[-(i+1)] or inc
            # Channels for skip: current_ch // 2
            # Channels for x_up (from previous decoder stage or transformer): current_ch
            # So, DoubleConv input for Up module: current_ch + (current_ch // 2)
            # DoubleConv output for Up module: current_ch // 2
            self.decoder_ups.append(Up(current_ch + (current_ch // 2), current_ch // 2))
            current_ch //= 2
            
        self.final_conv = nn.Conv2d(current_ch, out_chans, kernel_size=1) # current_ch should be base_channels

    def forward(self, x):
        # Encoder path
        skip_connections = []
        
        enc_x = self.encoder_inc(x)
        skip_connections.append(enc_x) # H, base_channels
        
        for i, down_layer in enumerate(self.encoder_downs):
            enc_x = down_layer(enc_x)
            if i < len(self.encoder_downs) -1: # Store all but the last output which goes to transformer
                skip_connections.append(enc_x)
        # enc_x is now the input to the transformer bottleneck, shape (B, C_transformer, H_bottleneck, W_bottleneck)

        # Transformer bottleneck
        B, C, H_tr, W_tr = enc_x.shape
        # Reshape for Transformer: (B, C, H, W) -> (B, H*W, C)
        tr_input = rearrange(enc_x, 'b c h w -> b (h w) c')
        tr_output = self.transformer_bottleneck(tr_input)
        # Reshape back: (B, H*W, C) -> (B, C, H, W)
        dec_x = rearrange(tr_output, 'b (h w) c -> b c h w', h=H_tr, w=W_tr)

        # Decoder path
        # Skips are from shallowest to deepest: skip_connections[0] is from inc, skip_connections[-1] is from second to last down
        # We need them in reverse for the Up layers: deepest skip first
        for i, up_layer in enumerate(self.decoder_ups):
            skip = skip_connections[-(i + 1)] # Correct skip connection
            dec_x = up_layer(dec_x, skip)
            
        out = self.final_conv(dec_x)
        
        # Ensure output size matches input size if necessary
        if out.shape[-2:] != x.shape[-2:]:
            out = F.interpolate(out, size=x.shape[-2:], mode='bilinear', align_corners=False)
            
        return out

# Visualization Functions

In [142]:
# Cell 6: Training, Validation, and Visualization Functions
# Cell 7: Training, Validation, and Visualization Functions (Adapted from Swin UNET Overfit.ipynb)

def train_epoch(model, dataloader, optimizer, criterion, device, scaler, accumulation_steps=1):
    model.train()
    running_loss = 0.0
    
    optimizer.zero_grad() 
    
    with tqdm(dataloader, desc="Training", leave=False) as pbar:
        for i, (inputs, targets) in enumerate(pbar):
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            if inputs.ndim == 3: inputs = inputs.unsqueeze(1)
            if targets.ndim == 3: targets = targets.unsqueeze(1)

            with autocast(device_type=device.type, enabled=scaler.is_enabled()): 
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                if accumulation_steps > 1:
                    loss = loss / accumulation_steps
            
            scaler.scale(loss).backward()
            
            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(dataloader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad() 

            current_loss = loss.item() * (accumulation_steps if (i + 1) % accumulation_steps != 0 and accumulation_steps > 1 else 1) # Correct loss scaling for logging
            running_loss += current_loss * inputs.size(0) # Weighted by batch size for epoch loss
            pbar.set_postfix({'loss': current_loss, 'lr': optimizer.param_groups[0]['lr']}) # Log true batch loss
            
    return running_loss / len(dataloader.dataset) # Average loss over all samples


def validate(model, dataloader, criterion, device, data_range_metrics=1.0):
    model.eval()
    running_loss = 0.0
    running_psnr = 0.0
    all_ssims = [] 
    
    with torch.no_grad():
        with tqdm(dataloader, desc="Validation", leave=False) as pbar:
            for inputs, targets in pbar:
                inputs = inputs.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)
                
                if inputs.ndim == 3: inputs = inputs.unsqueeze(1)
                if targets.ndim == 3: targets = targets.unsqueeze(1)
                
                outputs = model(inputs) 
                loss = criterion(outputs, targets)
                
                psnr_batch = calculate_psnr_torch(outputs, targets, data_range=data_range_metrics)
                
                running_loss += loss.item() * inputs.size(0) # Weighted by batch size
                # Accumulate PSNR weighted by batch size for correct averaging
                # Handle inf PSNR by capping or deciding on a large finite value if it affects averaging
                # For now, sum up finite PSNRs and count them
                finite_psnr_mask = ~torch.isinf(psnr_batch)
                if finite_psnr_mask.any():
                     running_psnr += psnr_batch[finite_psnr_mask].sum().item() * inputs.size(0) # This is not quite right. Should be per-image then averaged.
                # Let's average PSNR per batch, then average batch PSNRs. Or sum all PSNRs and divide by total images.
                # Simpler: calculate PSNR for each image in batch, then average.
                
                current_batch_psnr_sum = 0
                num_valid_psnr = 0
                for k_idx in range(outputs.size(0)):
                    psnr_single = calculate_psnr_torch(outputs[k_idx:k_idx+1], targets[k_idx:k_idx+1], data_range=data_range_metrics)
                    if not torch.isinf(psnr_single):
                        current_batch_psnr_sum += psnr_single.item()
                        num_valid_psnr +=1
                if num_valid_psnr > 0:
                    running_psnr += (current_batch_psnr_sum / num_valid_psnr) * inputs.size(0) # Weighted sum of batch average PSNRs


                for k in range(outputs.size(0)):
                    output_np = outputs[k].cpu().squeeze().numpy()
                    target_np = targets[k].cpu().squeeze().numpy()
                    all_ssims.append(calculate_ssim_skimage(output_np, target_np, data_range=data_range_metrics))
                    
                pbar.set_postfix({'val_loss': loss.item(), 
                                  'batch_avg_psnr': (current_batch_psnr_sum / num_valid_psnr if num_valid_psnr > 0 else float('nan'))})
                
    avg_loss = running_loss / len(dataloader.dataset)
    avg_psnr = running_psnr / len(dataloader.dataset) if len(dataloader.dataset) > 0 else 0
    avg_ssim = np.mean(all_ssims) if all_ssims else 0
    return avg_loss, avg_psnr, avg_ssim


def visualize_results(model, dataloader, device, epoch, output_dir_str, num_images=4, data_range_metrics=1.0):
    output_dir = Path(output_dir_str)
    output_dir.mkdir(parents=True, exist_ok=True)
    model.eval()
    
    try:
        inputs, targets = next(iter(dataloader))
    except StopIteration:
        print("Visualizer: Dataloader empty, skipping visualization.")
        return

    inputs = inputs.to(device)
    targets = targets.to(device)
    
    with torch.no_grad():
        outputs = model(inputs.unsqueeze(1) if inputs.ndim == 3 else inputs)
    
    num_to_show = min(num_images, inputs.size(0))
    if num_to_show == 0:
        print("Visualizer: No images to show from batch.")
        return

    fig, axes = plt.subplots(num_to_show, 3, figsize=(15, 5 * num_to_show))
    if num_to_show == 1: axes = np.array([axes]).reshape(1,3) # Ensure axes is 2D for single image
    
    model_name = model.__class__.__name__
    fig.suptitle(f"{model_name} Reconstruction - Epoch {epoch}", fontsize=16)
    
    for i in range(num_to_show): 
        input_img = inputs[i].cpu().squeeze().numpy()
        output_img = outputs[i].cpu().squeeze().numpy() 
        target_img = targets[i].cpu().squeeze().numpy()
            
        axes[i, 0].imshow(input_img, cmap='gray', vmin=0, vmax=data_range_metrics)
        axes[i, 0].set_title(f"Input (Undersampled)")
        axes[i, 0].axis('off')
            
        axes[i, 1].imshow(output_img, cmap='gray', vmin=0, vmax=data_range_metrics)
        axes[i, 1].set_title(f"Prediction")
        axes[i, 1].axis('off')
            
        axes[i, 2].imshow(target_img, cmap='gray', vmin=0, vmax=data_range_metrics)
        axes[i, 2].set_title(f"Ground Truth")
        axes[i, 2].axis('off')
        
        psnr_val = calculate_psnr_torch(
            outputs[i].unsqueeze(0), 
            targets[i].unsqueeze(0),
            data_range=data_range_metrics
        ).item()
        ssim_val = calculate_ssim_skimage(output_img, target_img, data_range=data_range_metrics)
            
        axes[i, 1].text(
            5, 15, 
            f'PSNR: {psnr_val:.2f}\nSSIM: {ssim_val:.4f}',
            color='white', fontsize=10, 
            bbox=dict(facecolor='black', alpha=0.5)
        )
    
    plt.tight_layout(rect=[0, 0, 1, 0.96]) 
    plt.savefig(output_dir / f"epoch_{epoch}_visualization.png")
    plt.close(fig)

In [143]:
from pathlib import Path # Ensure Path is imported in this cell or globally

# ... (assuming _orig_init is correctly defined from the non-patched ProcessedFastMRIDataset.__init__)

def _patched_init(self, *args, **kwargs):
    _orig_init(self, *args, **kwargs)

    good_examples = []
    if hasattr(self, 'batch_files') and self.batch_files: # Check if batch_files exists and is not empty
        for idx, slice_idx in self.examples:
            # Ensure idx is a valid index for self.batch_files
            if 0 <= idx < len(self.batch_files):
                batch_file_string_path = self.batch_files[idx]
                # Convert the string path to a Path object
                path_obj = Path(batch_file_string_path)
                if "metadata" not in path_obj.name:  # Now path_obj.name is correct
                    good_examples.append((idx, slice_idx))
            else:
                print(f"Warning in patch: Invalid index {idx} for batch_files of length {len(self.batch_files)}")
        self.examples = good_examples
    elif not hasattr(self, 'batch_files') or not self.batch_files:
        # This case might occur if _orig_init failed to populate self.batch_files
        # or if use_processed was False (though the error trace suggests use_processed=True)
        print("Warning in patch: self.batch_files not populated or empty, skipping example filtering.")
        # self.examples would remain as whatever _orig_init set it to, or cause error if not set.

# ProcessedFastMRIDataset.__init__ = _patched_init # This line applies the patch
# print("Patched ProcessedFastMRIDataset to ignore files that contain 'metadata'")


# Main Training Script

In [144]:
# Cell 8: Main Training Script for BT-UNet

def run_tuning_experiment(run_name, model_fn, train_ds, val_ds,
                          batch_size=16, epochs=50, lr=1e-4, frac=1.0,
                          viz_every=10, out_root_str="./runs_bt_unet", 
                          device_obj=None, accumulation_steps=1,
                          weight_decay=1e-4, lr_patience=5, lr_factor=0.5,
                          ssim_alpha=0.84, ssim_data_range=1.0, metrics_data_range=1.0):

    out_dir = Path(out_root_str) / run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    current_device = device_obj if device_obj else torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Dataloader definition (local to this function to use its params)
    def make_loader_local(ds, shuffle, current_frac=1.0, current_bs=batch_size):
        if not ds or len(ds) == 0: # Handle empty dataset
             print(f"Warning: Dataset for {'train' if shuffle else 'val'} is empty or None.")
             # Return an empty dataloader or a dummy one if preferred
             return DataLoader(Subset(ds, []), batch_size=current_bs, shuffle=False)


        if current_frac < 1.0:
            n = int(len(ds) * current_frac)
            if n == 0 and len(ds) > 0: n = 1 # Ensure at least one sample
            # Ensure 'n' isn't larger than the dataset length
            n = min(n, len(ds))
            idx = random.sample(range(len(ds)), n) if len(ds) > n else list(range(len(ds)))
            ds_subset = Subset(ds, idx)
        else:
            ds_subset = ds
        
        if len(ds_subset) == 0:
            print(f"Warning: DataLoader for {'train' if shuffle else 'val'} created with 0 samples after subsetting.")
            return DataLoader(ds_subset, batch_size=current_bs, shuffle=shuffle, num_workers=0, pin_memory=True) # num_workers to 0 for empty

        return DataLoader(ds_subset, batch_size=current_bs,
                          shuffle=shuffle, num_workers=min(4, os.cpu_count()//2 if os.cpu_count() else 1), # Robust num_workers
                          pin_memory=True, drop_last=(True if shuffle and len(ds_subset) > current_bs else False))

    train_loader = make_loader_local(train_ds, True,  frac)
    val_loader   = make_loader_local(val_ds,   False, 1.0) # Validate on full validation set

    if len(train_loader.dataset) == 0:
        print(f"Skipping run {run_name} as the training dataloader is empty.")
        return run_name, float('inf'), 0, 0


    model = model_fn().to(current_device)
    criterion = CombinedLoss(alpha=ssim_alpha, ssim_data_range=ssim_data_range).to(current_device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=lr_factor, patience=lr_patience)
    scaler = GradScaler(device=current_device.type, enabled=current_device.type=='cuda')

    best_val_loss = float('inf')
    history = [] 

    print(f"\n▶ Starting Run: {run_name}")
    print(f"  Config: LR={lr}, Epochs={epochs}, Batch={batch_size}, AccumSteps={accumulation_steps}")
    print(f"  Training on {len(train_loader.dataset)} slices ({frac*100:.0f}% of train set). Validating on {len(val_loader.dataset)} slices.")
    print(f"  Model: {model.__class__.__name__}, Device: {current_device.type}")
    print(f"  Saving to: {out_dir}")

    for ep in range(1, epochs + 1):
        epoch_start_time = time.time()
        print(f"\n{run_name} | Epoch {ep}/{epochs}")

        tr_loss = train_epoch(model, train_loader, optimizer, criterion, current_device, scaler, accumulation_steps)
        val_loss, val_psnr, val_ssim = validate(model, val_loader, criterion, current_device, data_range_metrics=metrics_data_range)
        scheduler.step(val_loss)
        
        epoch_duration = time.time() - epoch_start_time
        history.append([ep, tr_loss, val_loss, val_psnr, val_ssim, optimizer.param_groups[0]['lr'], epoch_duration])
        print(f"  Epoch {ep} Stats: Train Loss: {tr_loss:.4f}, Val Loss: {val_loss:.4f}, Val PSNR: {val_psnr:.2f}, Val SSIM: {val_ssim:.4f}, LR: {optimizer.param_groups[0]['lr']:.2e}, Time: {epoch_duration:.2f}s")

        if ep % viz_every == 0 or ep == epochs:
            if len(val_loader.dataset) > 0:
                 visualize_results(model, val_loader, current_device, epoch=ep, output_dir_str=str(out_dir), data_range_metrics=metrics_data_range)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'epoch': ep, 
                'model_state_dict': model.state_dict(), 
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': best_val_loss,
                'val_psnr': val_psnr,
                'val_ssim': val_ssim
            }, out_dir / 'best_model.pth')
            print(f"  Saved new best model at epoch {ep} with Val Loss: {best_val_loss:.4f}")

        if ep % 10 == 0: # Checkpoint every 10 epochs
            torch.save({
                'epoch': ep, 
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
            }, out_dir / f'ckpt_epoch_{ep}.pth')

    hist_np = np.asarray(history)
    if hist_np.size > 0:
        header_str = "epoch,train_loss,val_loss,psnr,ssim,lr,epoch_time_s"
        np.savetxt(out_dir / "history.csv", hist_np, delimiter=',', header=header_str, comments='')

        plt.figure(figsize=(18, 10))
        plt.subplot(2, 2, 1)
        plt.plot(hist_np[:,0], hist_np[:,1], label='Train Loss')
        plt.plot(hist_np[:,0], hist_np[:,2], label='Val Loss')
        plt.title(f"Loss ({run_name})"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(True)
        
        plt.subplot(2, 2, 2)
        plt.plot(hist_np[:,0], hist_np[:,3])
        plt.title(f"PSNR ({run_name})"); plt.xlabel("Epoch"); plt.ylabel("PSNR (dB)"); plt.grid(True)
        
        plt.subplot(2, 2, 3)
        plt.plot(hist_np[:,0], hist_np[:,4])
        plt.title(f"SSIM ({run_name})"); plt.xlabel("Epoch"); plt.ylabel("SSIM"); plt.grid(True)

        plt.subplot(2, 2, 4)
        plt.plot(hist_np[:,0], hist_np[:,5])
        plt.title(f"Learning Rate ({run_name})"); plt.xlabel("Epoch"); plt.ylabel("LR"); plt.yscale('log'); plt.grid(True)
        
        plt.tight_layout()
        plt.savefig(out_dir / "training_curves.png")
        plt.close()
    else:
        print(f"Warning: History for {run_name} is empty. No curves plotted.")

    print(f"▶ Finished Run: {run_name}. Best Val Loss: {best_val_loss:.4f}")
    # Return last recorded metrics for simplicity, or load best model and re-evaluate
    final_psnr_val = history[-1][3] if history else 0
    final_ssim_val = history[-1][4] if history else 0
    
    return run_name, best_val_loss, final_psnr_val, final_ssim_val, str(out_dir / 'best_model.pth')


# Inference and Model Evaluation

In [145]:
def evaluate_model(model_path, dataloader, device, output_dir="./evaluation"):
    """Evaluate a trained model on a dataset"""
    # Load the model
    model = UNet(in_chans=1, out_chans=1, chans=32, num_pool_layers=4).to(device)
    
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Initialize metrics
    psnrs = []
    ssims = []
    
    # Process batches
    with torch.no_grad():
        for i, (inputs, targets) in enumerate(tqdm(dataloader, desc="Evaluating")):
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            # Add channel dimension if needed
            if len(inputs.shape) == 3:
                inputs = inputs.unsqueeze(1)
            if len(targets.shape) == 3:
                targets = targets.unsqueeze(1)
            
            # Make predictions
            outputs = model(inputs)
            
            # Calculate metrics
            for j in range(outputs.size(0)):
                output_np = outputs[j, 0].cpu().numpy()
                target_np = targets[j, 0].cpu().numpy()
                
                psnr = calculate_psnr(outputs[j:j+1], targets[j:j+1]).item()
                ssim_val = calculate_ssim(output_np, target_np)
                
                psnrs.append(psnr)
                ssims.append(ssim_val)
            
            # Visualize first batch
            if i == 0:
                fig, axes = plt.subplots(4, 3, figsize=(15, 20))
                fig.suptitle("FastMRI Reconstruction Results", fontsize=16)
                
                for j in range(min(4, outputs.size(0))):
                    # Get images
                    input_img = inputs[j, 0].cpu().numpy()
                    output_img = outputs[j, 0].cpu().numpy()
                    target_img = targets[j, 0].cpu().numpy()
                    
                    # Display input
                    axes[j, 0].imshow(input_img, cmap='gray')
                    axes[j, 0].set_title(f"Input (Undersampled)")
                    axes[j, 0].axis('off')
                    
                    # Display output
                    axes[j, 1].imshow(output_img, cmap='gray')
                    axes[j, 1].set_title(f"Prediction")
                    axes[j, 1].axis('off')
                    
                    # Display target
                    axes[j, 2].imshow(target_img, cmap='gray')
                    axes[j, 2].set_title(f"Ground Truth")
                    axes[j, 2].axis('off')
                    
                    # Add metrics as text
                    axes[j, 1].text(
                        10, 20, 
                        f'PSNR: {psnrs[j]:.2f} dB\nSSIM: {ssims[j]:.4f}',
                        color='white', fontsize=12, 
                        bbox=dict(facecolor='black', alpha=0.5)
                    )
                
                plt.tight_layout()
                plt.subplots_adjust(top=0.95)
                plt.savefig(f"{output_dir}/evaluation_samples.png")
                plt.close()
    
    # Calculate average metrics
    avg_psnr = np.mean(psnrs)
    avg_ssim = np.mean(ssims)
    
    # Print results
    print(f"Average PSNR: {avg_psnr:.2f} dB")
    print(f"Average SSIM: {avg_ssim:.4f}")
    
    # Save metrics
    results = {
        'psnr': psnrs,
        'ssim': ssims,
        'avg_psnr': avg_psnr,
        'avg_ssim': avg_ssim
    }
    
    # Save in text file
    with open(f"{output_dir}/evaluation_results.txt", 'w') as f:
        f.write(f"U-Net Baseline Evaluation Results\n")
        f.write(f"Average PSNR: {avg_psnr:.2f} dB\n")
        f.write(f"Average SSIM: {avg_ssim:.4f}\n")
    
    # Plot histograms of metrics
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.hist(psnrs, bins=20)
    plt.xlabel('PSNR (dB)')
    plt.ylabel('Count')
    plt.title(f'PSNR Histogram (Avg: {avg_psnr:.2f} dB)')
    
    plt.subplot(1, 2, 2)
    plt.hist(ssims, bins=20)
    plt.xlabel('SSIM')
    plt.ylabel('Count')
    plt.title(f'SSIM Histogram (Avg: {avg_ssim:.4f})')
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/metric_histograms.png")
    plt.close()
    
    return results

# Example usage (uncomment to run)
# val_dataset = ProcessedFastMRIDataset(data_dir="./processed_fastmri_data", mode='val', use_processed=True)
# val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4)
# results = evaluate_model("./unet_output/best_model.pth", val_loader, device)


# Run

In [146]:
if __name__ == "__main__":
    DATA_ROOT = Path("/workspace/fastmri-reconstruction/processed_fastmri_data") # Adjust if your path is different
    OUTPUT_ROOT_UNET_SINGLE = Path("./unet_baseline_single_experiment")
    OUTPUT_ROOT_UNET_SINGLE.mkdir(parents=True, exist_ok=True)

    print(f"Attempting to load data from: {DATA_ROOT}")
    try:
        print("Loading full datasets for U-Net baseline...")
        train_dataset = ProcessedFastMRIDataset(data_dir=DATA_ROOT, mode='train', use_processed=True)
        val_dataset   = ProcessedFastMRIDataset(data_dir=DATA_ROOT, mode='val',   use_processed=True)
        print(f"Successfully loaded {len(train_dataset)} training samples and {len(val_dataset)} validation samples.")
    except Exception as e:
        print(f"Critical Error loading datasets: {e}")
        print("Please ensure the DATA_ROOT path is correct and data is preprocessed as .pt files.")
        print("Script will exit as datasets are crucial.")
        raise SystemExit("Dataset loading failed for U-Net baseline.")

    # Single U-Net Baseline Configuration
    # Parameters for UNet: chans, num_pool_layers
    unet_single_config = {
        'name_suffix': 'chans32_l4_lr1e4_bs16_epochs50_wd1e4_baseline', # Descriptive suffix
        'chans': 32, 
        'num_pool_layers': 4,
        'lr': 1e-4, 
        'epochs': 40, # Adjusted epochs for a single focused run
        'batch_size': 16, 
        'acc_steps': 1,
        'weight_decay': 1e-4, 
        'frac': 1.0  # Use full dataset
    }
    
    experiment_results_list = [] # Renamed to avoid confusion if other cells use 'experiment_results'
    
    config = unet_single_config # Use the single config
    run_name = f"UNet_{config['name_suffix']}"
    
    model_lambda = lambda cfg=config: UNet( # Use UNet class directly
        in_chans=1, out_chans=1,
        chans=cfg['chans'],
        num_pool_layers=cfg['num_pool_layers']
    )
    
    if len(train_dataset) == 0 or len(val_dataset) == 0:
        print(f"Skipping run {run_name} due to empty dataset(s). Check data loading.")
    else:
        try:
            print(f"--- Preparing for U-Net baseline run: {run_name} ---")
            res_name, res_loss, res_psnr, res_ssim, res_path = run_tuning_experiment(
                run_name=run_name,
                model_fn=model_lambda,
                train_ds=train_dataset,
                val_ds=val_dataset,
                batch_size=config['batch_size'],
                epochs=config['epochs'],
                lr=config['lr'],
                frac=config['frac'],
                viz_every=10, # Visualize every 10 epochs, or as preferred
                out_root_str=str(OUTPUT_ROOT_UNET_SINGLE), # Save to U-Net specific directory
                device_obj=device, # Pass the global device object
                accumulation_steps=config['acc_steps'],
                weight_decay=config['weight_decay']
                # ssim_alpha, ssim_data_range, metrics_data_range will use defaults from run_tuning_experiment
            )
            experiment_results_list.append({
                'run_name': res_name,
                'config_suffix': config['name_suffix'],
                'best_val_loss': res_loss,
                'final_val_psnr': res_psnr,
                'final_val_ssim': res_ssim,
                'model_path': res_path,
                **config # Add all config params to results for easy review
            })
        except Exception as e:
            print(f"Error during U-Net training for {run_name}: {e}")
            import traceback
            traceback.print_exc()
            experiment_results_list.append({
                'run_name': run_name,
                'config_suffix': config['name_suffix'],
                'best_val_loss': float('inf'), 'final_val_psnr': 0, 'final_val_ssim': 0,
                'model_path': 'Error',
                 **config
            })

    # Display U-Net results
    if experiment_results_list:
        unet_results_df = pd.DataFrame(experiment_results_list)
        # No need to sort by best_val_loss if there's only one run, but good practice if extending later
        unet_results_df = unet_results_df.sort_values(by="best_val_loss", ascending=True)
        print("\n--- U-Net Baseline Experiment Result ---")
        
        display_cols_unet = ['run_name', 'best_val_loss', 'final_val_psnr', 'final_val_ssim', 
                             'chans', 'num_pool_layers', 
                             'lr', 'batch_size', 'acc_steps', 'epochs', 'model_path']
        # Filter out columns that might not exist if there was an error early
        display_cols_unet = [col for col in display_cols_unet if col in unet_results_df.columns]

        # Print to console (to_string() provides better formatting for wide tables)
        print(unet_results_df[display_cols_unet].to_string())
        
        # Save to CSV
        summary_csv_path = OUTPUT_ROOT_UNET_SINGLE / "unet_baseline_single_experiment_summary.csv"
        unet_results_df.to_csv(summary_csv_path, index=False)
        print(f"\nFull U-Net result saved to: {summary_csv_path}")
    else:
        print("The U-Net baseline experiment was not completed or did not produce results.")




Attempting to load data from: /workspace/fastmri-reconstruction/processed_fastmri_data
Loading full datasets for U-Net baseline...
Successfully indexed a total of 973 examples from 61 .pt files in /workspace/fastmri-reconstruction/processed_fastmri_data/train.
Successfully indexed a total of 199 examples from 13 .pt files in /workspace/fastmri-reconstruction/processed_fastmri_data/val.
Successfully loaded 973 training samples and 199 validation samples.
--- Preparing for U-Net baseline run: UNet_chans32_l4_lr1e4_bs16_epochs50_wd1e4_baseline ---

▶ Starting Run: UNet_chans32_l4_lr1e4_bs16_epochs50_wd1e4_baseline
  Config: LR=0.0001, Epochs=40, Batch=16, AccumSteps=1
  Training on 973 slices (100% of train set). Validating on 199 slices.
  Model: UNet, Device: cuda
  Saving to: unet_baseline_single_experiment/UNet_chans32_l4_lr1e4_bs16_epochs50_wd1e4_baseline

UNet_chans32_l4_lr1e4_bs16_epochs50_wd1e4_baseline | Epoch 1/40


Training:   0%|          | 0/60 [00:00<?, ?it/s]

Validation:   0%|          | 0/13 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x77beb80cd3a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1663, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1646, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.11/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x77beb80cd3a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1663, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 16

  Epoch 1 Stats: Train Loss: 0.1235, Val Loss: 0.0757, Val PSNR: 49.77, Val SSIM: 0.7276, LR: 1.00e-04, Time: 4.83s
  Saved new best model at epoch 1 with Val Loss: 0.0757

UNet_chans32_l4_lr1e4_bs16_epochs50_wd1e4_baseline | Epoch 2/40


...

Validation:   0%|          | 0/13 [00:00<?, ?it/s]

  Epoch 39 Stats: Train Loss: 0.0498, Val Loss: 0.0496, Val PSNR: 58.23, Val SSIM: 0.8266, LR: 5.00e-05, Time: 4.92s
  Saved new best model at epoch 39 with Val Loss: 0.0496

UNet_chans32_l4_lr1e4_bs16_epochs50_wd1e4_baseline | Epoch 40/40


Training:   0%|          | 0/60 [00:00<?, ?it/s]

Validation:   0%|          | 0/13 [00:00<?, ?it/s]

  Epoch 40 Stats: Train Loss: 0.0499, Val Loss: 0.0502, Val PSNR: 58.01, Val SSIM: 0.8255, LR: 5.00e-05, Time: 4.91s
▶ Finished Run: UNet_chans32_l4_lr1e4_bs16_epochs50_wd1e4_baseline. Best Val Loss: 0.0496

--- U-Net Baseline Experiment Result ---
                                             run_name  best_val_loss  final_val_psnr  final_val_ssim  chans  num_pool_layers      lr  batch_size  acc_steps  epochs                                                                                         model_path
0  UNet_chans32_l4_lr1e4_bs16_epochs50_wd1e4_baseline        0.04962       58.010789         0.82554     32                4  0.0001          16          1      40  unet_baseline_single_experiment/UNet_chans32_l4_lr1e4_bs16_epochs50_wd1e4_baseline/best_model.pth

Full U-Net result saved to: unet_baseline_single_experiment/unet_baseline_single_experiment_summary.csv


In [149]:

if __name__ == "__main__":
    DATA_ROOT = Path("/workspace/fastmri-reconstruction/processed_fastmri_data") # Adjust if your path is different
    OUTPUT_ROOT = Path("./runs_bt_unet_experiment")
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    print(f"Attempting to load data from: {DATA_ROOT}")
    try:
        # For faster debugging, use a smaller fraction of data initially
        # train_dataset = ProcessedFastMRIDataset(data_dir=DATA_ROOT, mode='train', use_processed=True, target_size=(320,320))
        # val_dataset   = ProcessedFastMRIDataset(data_dir=DATA_ROOT, mode='val', use_processed=True, target_size=(320,320))
        
        # Use full dataset for actual runs
        # Create dummy datasets for now if data loading is slow or problematic for initial script run
        # This is just to ensure the script runs through. Replace with actual data loading.
        print("Loading full datasets...")
        train_dataset = ProcessedFastMRIDataset(data_dir=DATA_ROOT, mode='train', use_processed=True)
        val_dataset   = ProcessedFastMRIDataset(data_dir=DATA_ROOT, mode='val',   use_processed=True)
        print(f"Successfully loaded {len(train_dataset)} training samples and {len(val_dataset)} validation samples.")
    except Exception as e:
        print(f"Critical Error loading datasets: {e}")
        print("Please ensure the DATA_ROOT path is correct and data is preprocessed as .pt files.")
        print("Script will exit as datasets are crucial.")
        raise SystemExit("Dataset loading failed.")

    # BT-UNet Hyperparameter Configurations
    # Parameters for BTUNet: base_channels, num_pool_layers, tr_depth, tr_heads, tr_dropout
    bt_unet_configs = [
        {
            'name_suffix': 'bc32_l4_d4_h8_dr01_lr1e4_bs8_wd1e4',
            'base_channels': 32, 'num_pool_layers': 4, 
            'tr_depth': 4, 'tr_heads': 8, 'tr_dropout': 0.1,
            'lr': 1e-4, 'epochs': 40, 'batch_size': 8, 'acc_steps': 1,
            'weight_decay': 1e-4, 'frac': 1.0 
        },
        {
            'name_suffix': 'bc32_l4_d6_h8_dr01_lr1e4_bs8_wd1e4', # Deeper transformer
            'base_channels': 32, 'num_pool_layers': 4,
            'tr_depth': 6, 'tr_heads': 8, 'tr_dropout': 0.1,
            'lr': 1e-4, 'epochs': 40, 'batch_size': 8, 'acc_steps': 1,
            'weight_decay': 1e-4, 'frac': 1.0
        },
        {
            'name_suffix': 'bc64_l4_d4_h8_dr01_lr1e4_bs4_acc2_wd1e4', # Wider UNet, smaller batch + accum
            'base_channels': 64, 'num_pool_layers': 4,
            'tr_depth': 4, 'tr_heads': 8, 'tr_dropout': 0.1,
            'lr': 1e-4, 'epochs': 40, 'batch_size': 4, 'acc_steps': 2, # Effective BS = 8
            'weight_decay': 1e-4, 'frac': 1.0
        },
        {
            'name_suffix': 'bc32_l4_d4_h8_dr01_lr5e4_bs8_wd1e4', # Higher LR
            'base_channels': 32, 'num_pool_layers': 4,
            'tr_depth': 4, 'tr_heads': 8, 'tr_dropout': 0.1,
            'lr': 5e-4, 'epochs': 40, 'batch_size': 8, 'acc_steps': 1,
            'weight_decay': 1e-4, 'frac': 1.0
        },
         {
            'name_suffix': 'bc32_l3_d4_h8_dr01_lr1e4_bs8_wd1e4', # Shallower UNet (fewer pool layers)
            'base_channels': 32, 'num_pool_layers': 3, # Transformer dim will be smaller
            'tr_depth': 4, 'tr_heads': 8, 'tr_dropout': 0.1,
            'lr': 1e-4, 'epochs': 40, 'batch_size': 8, 'acc_steps': 1,
            'weight_decay': 1e-4, 'frac': 1.0
        },
    ]
    
    experiment_results = []
    
    for config in bt_unet_configs:
        run_name = f"BTUNet_{config['name_suffix']}"
        
        model_lambda = lambda cfg=config: BTUNet(
            in_chans=1, out_chans=1,
            base_channels=cfg['base_channels'],
            num_pool_layers=cfg['num_pool_layers'],
            tr_depth=cfg['tr_depth'],
            tr_heads=cfg['tr_heads'],
            tr_dropout=cfg['tr_dropout']
        )
        
        if len(train_dataset) == 0 or len(val_dataset) == 0:
            print(f"Skipping run {run_name} due to empty dataset(s). Check data loading.")
            continue
    
        try:
            print(f"--- Preparing for run: {run_name} ---")
            res_name, res_loss, res_psnr, res_ssim, res_path = run_tuning_experiment(
                run_name=run_name,
                model_fn=model_lambda,
                train_ds=train_dataset,
                val_ds=val_dataset,
                batch_size=config['batch_size'],
                epochs=config['epochs'],
                lr=config['lr'],
                frac=config['frac'],
                viz_every=10, 
                out_root_str=str(OUTPUT_ROOT),
                device_obj=device, # Pass the global device object
                accumulation_steps=config['acc_steps'],
                weight_decay=config['weight_decay']
            )
            experiment_results.append({
                'run_name': res_name,
                'config_suffix': config['name_suffix'],
                'best_val_loss': res_loss,
                'final_val_psnr': res_psnr,
                'final_val_ssim': res_ssim,
                'model_path': res_path,
                **config # Add all config params to results for easy review
            })
        except Exception as e:
            print(f"Error during training for {run_name}: {e}")
            import traceback
            traceback.print_exc()
            experiment_results.append({
                'run_name': run_name,
                'config_suffix': config['name_suffix'],
                'best_val_loss': float('inf'), 'final_val_psnr': 0, 'final_val_ssim': 0,
                'model_path': 'Error',
                 **config
            })

    # Display results
    if experiment_results:
        results_df = pd.DataFrame(experiment_results)
        results_df = results_df.sort_values(by="best_val_loss", ascending=True)
        print("\n--- BT-UNet Experiment Results Summary ---")
        
        # Select key columns for display, or display all
        display_cols = ['run_name', 'best_val_loss', 'final_val_psnr', 'final_val_ssim', 
                        'base_channels', 'num_pool_layers', 'tr_depth', 'tr_heads', 
                        'lr', 'batch_size', 'acc_steps', 'epochs']
        # Filter out columns that might not exist if there was an error early
        display_cols = [col for col in display_cols if col in results_df.columns]

        print(results_df[display_cols].to_string())
        results_df.to_csv(OUTPUT_ROOT / "bt_unet_experiment_summary.csv", index=False)
        print(f"\nFull results saved to: {OUTPUT_ROOT / 'bt_unet_experiment_summary.csv'}")
    else:
        print("No BT-UNet experiments were completed.")


Attempting to load data from: /workspace/fastmri-reconstruction/processed_fastmri_data
Loading full datasets...
Successfully indexed a total of 973 examples from 61 .pt files in /workspace/fastmri-reconstruction/processed_fastmri_data/train.
Successfully indexed a total of 199 examples from 13 .pt files in /workspace/fastmri-reconstruction/processed_fastmri_data/val.
Successfully loaded 973 training samples and 199 validation samples.
--- Preparing for run: BTUNet_bc32_l4_d4_h8_dr01_lr1e4_bs8_wd1e4 ---

▶ Starting Run: BTUNet_bc32_l4_d4_h8_dr01_lr1e4_bs8_wd1e4
  Config: LR=0.0001, Epochs=40, Batch=8, AccumSteps=1
  Training on 973 slices (100% of train set). Validating on 199 slices.
  Model: BTUNet, Device: cuda
  Saving to: runs_bt_unet_experiment/BTUNet_bc32_l4_d4_h8_dr01_lr1e4_bs8_wd1e4

BTUNet_bc32_l4_d4_h8_dr01_lr1e4_bs8_wd1e4 | Epoch 1/40


Training:   0%|          | 0/121 [00:00<?, ?it/s]

Validation:   0%|          | 0/25 [00:00<?, ?it/s]

  Epoch 1 Stats: Train Loss: 0.1139, Val Loss: 0.0717, Val PSNR: 51.05, Val SSIM: 0.7427, LR: 1.00e-04, Time: 6.43s
  Saved new best model at epoch 1 with Val Loss: 0.0717

BTUNet_bc32_l4_d4_h8_dr01_lr1e4_bs8_wd1e4 | Epoch 2/40


Training:   0%|          | 0/121 [00:00<?, ?it/s]

...

  Epoch 39 Stats: Train Loss: 0.0469, Val Loss: 0.0510, Val PSNR: 57.65, Val SSIM: 0.8269, LR: 5.00e-05, Time: 8.19s

BTUNet_bc32_l3_d4_h8_dr01_lr1e4_bs8_wd1e4 | Epoch 40/40


Training:   0%|          | 0/121 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x77beb80cd3a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1663, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1646, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.11/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x77beb80cd3a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1663, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 16

Validation:   0%|          | 0/25 [00:00<?, ?it/s]

  Epoch 40 Stats: Train Loss: 0.0467, Val Loss: 0.0499, Val PSNR: 58.58, Val SSIM: 0.8275, LR: 5.00e-05, Time: 9.26s
▶ Finished Run: BTUNet_bc32_l3_d4_h8_dr01_lr1e4_bs8_wd1e4. Best Val Loss: 0.0491

--- BT-UNet Experiment Results Summary ---
                                         run_name  best_val_loss  final_val_psnr  final_val_ssim  base_channels  num_pool_layers  tr_depth  tr_heads      lr  batch_size  acc_steps  epochs
3       BTUNet_bc32_l4_d4_h8_dr01_lr5e4_bs8_wd1e4       0.045040       58.098346        0.829771             32                4         4         8  0.0005           8          1      40
2  BTUNet_bc64_l4_d4_h8_dr01_lr1e4_bs4_acc2_wd1e4       0.048260       58.257792        0.830875             64                4         4         8  0.0001           4          2      40
4       BTUNet_bc32_l3_d4_h8_dr01_lr1e4_bs8_wd1e4       0.049130       58.575131        0.827501             32                3         4         8  0.0001           8          1      40
1     